# Stage 3 — Evaluation

Calculates precision, recall, and F1 for the thematic allocation at theme and sub-theme level.

**Runtime:** Google Colab + Google Drive  
**Storage:** All inputs and outputs are read from / written to a project folder on Google Drive.  
**Library:** [`multilingual-topic-modeling`](https://github.com/ay94/multilingual-topic-modeling)

**Inputs:** One or more evaluation CSVs with annotator decisions added (columns: `theme`, `subtheme`, `theme_decision`, `subtheme_decision`)

**Outputs:** Classification report per evaluation round

See [`docs/evaluation/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/evaluation) for the full methodology.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install multilingual-topic-modeling --quiet

In [ ]:
import numpy as np
import pandas as pd

from multilingual_topic import FileHandler, Evaluation

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/YOUR_PROJECT/Topic Modelling Workflow'

# Evaluation CSVs — add one entry per round
# Each file should have model-assigned labels and annotator decisions
EVAL_FILES = [
    'outputs/evaluation_sample_round1.csv',
    # 'outputs/evaluation_sample_round2.csv',
]

THEME_COL           = 'theme'             # model-assigned theme column
THEME_DECISION_COL  = 'theme_decision'    # annotator theme decision column
SUB_COL             = 'subtheme'          # model-assigned sub-theme column
SUB_DECISION_COL    = 'subtheme_decision' # annotator sub-theme decision column
RELEVANCY_COL       = 'relevancy'         # optional: filter to relevant messages only
# ──────────────────────────────────────────────────────────────────────────────

fh = FileHandler(DRIVE_FOLDER)

## 1. Load evaluation data

In [ ]:
rounds = [pd.read_csv(fh.create_filename(f)) for f in EVAL_FILES]
eval_df = pd.concat(rounds, ignore_index=True)
print(f'Evaluation rows: {len(eval_df):,}')
eval_df.head()

## 2. Theme-level evaluation

In [ ]:
# Filter to relevant messages if column exists
if RELEVANCY_COL in eval_df.columns:
    eval_relevant = eval_df[eval_df[RELEVANCY_COL] == 'Relevant'].copy()
else:
    eval_relevant = eval_df.copy()

print(f'Relevant messages for theme evaluation: {len(eval_relevant):,}')

In [ ]:
theme_eval = Evaluation(
    y_true=eval_relevant[THEME_DECISION_COL].values,
    y_pred=eval_relevant[THEME_COL].values,
)
print('=== Theme-level ===')
theme_eval.report()
theme_eval.confusion_matrix().show()

## 3. Sub-theme level evaluation

In [ ]:
sub_data = eval_relevant.dropna(subset=[SUB_DECISION_COL])

sub_eval = Evaluation(
    y_true=sub_data[SUB_DECISION_COL].values,
    y_pred=sub_data[SUB_COL].values,
)
print('=== Sub-theme level ===')
sub_eval.report()
sub_eval.confusion_matrix().show()

## 4. Summary metrics

In [ ]:
theme_metrics = theme_eval.metrics()
sub_metrics   = sub_eval.metrics()

summary = pd.DataFrame({
    'level':     ['theme', 'sub-theme'],
    'precision': [theme_metrics['precision'], sub_metrics['precision']],
    'recall':    [theme_metrics['recall'],    sub_metrics['recall']],
    'f1':        [theme_metrics['f1'],        sub_metrics['f1']],
    'n':         [len(eval_relevant),         len(sub_data)],
})
summary